<a href="https://colab.research.google.com/github/prashantbhoye2003-debug/Development-of-a-Healthcare-Operations-Intelligence-Dashboard-with-Decision-Analytics/blob/main/python_task.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

###<b>Python Task: Patient Attrition Analysis Dashboard Objective</b>

####<b>Prompt :</b> "Write a Python program using Pandas to create an patient dataset, display the data, calculate basic statistics (total patient, average age , average bill), and count of patient in each healthcare department."



In [3]:
import sqlite3
import pandas as pd

# 1. Connect to SQLite database (or create if it doesn't exist)
conn = sqlite3.connect('patients.db') # Changed database name
cursor = conn.cursor()

# 2. Create patients table
cursor.execute('''
    CREATE TABLE IF NOT EXISTS patients (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        name TEXT NOT NULL,
        age INTEGER NOT NULL,
        department TEXT NOT NULL,
        bill REAL NOT NULL
    )
''')
conn.commit()
print("Database and 'patients' table created successfully.")

# 3. Insert sample patient records
sample_patients = [
    ('John Doe', 35, 'Cardiology', 1500.50),
    ('Jane Smith', 50, 'Neurology', 3200.75),
    ('Peter Jones', 28, 'Healthcare', 800.00),
    ('Anna Lee', 65, 'Cardiology', 2100.20),
    ('Mike Brown', 42, 'Healthcare', 1200.00),
    ('Sarah Davis', 71, 'Oncology', 5500.00)
]

cursor.executemany("INSERT INTO patients (name, age, department, bill) VALUES (?, ?, ?, ?)", sample_patients)
conn.commit()
print("\nSample patient records inserted.")

# Function to fetch and display data using Pandas
def display_data(query, title):
    print(f"\n--- {title} ---")
    df = pd.read_sql_query(query, conn)
    print(df.to_markdown(index=False))

# 4. Perform SQL queries

# a. Display database statistics
print("\n--- Database Statistics ---")
# Total patients
total_patients = pd.read_sql_query("SELECT COUNT(*) FROM patients", conn).iloc[0, 0]
print(f"Total Patients: {total_patients}")

# Average age
average_age = pd.read_sql_query("SELECT AVG(age) FROM patients", conn).iloc[0, 0]
print(f"Average Age: {average_age:.2f} years")

# Average bill (added)
average_bill = pd.read_sql_query("SELECT AVG(bill) FROM patients", conn).iloc[0, 0]
print(f"Average Bill: ${average_bill:.2f}")

# Bill range (min and max bill)
bill_range = pd.read_sql_query("SELECT MIN(bill), MAX(bill) FROM patients", conn)
min_bill = bill_range.iloc[0, 0]
max_bill = bill_range.iloc[0, 1]
print(f"Bill Range: ${min_bill:.2f} - ${max_bill:.2f}")

# Count of patients in each department (added)
department_counts = pd.read_sql_query("SELECT department, COUNT(*) as patient_count FROM patients GROUP BY department", conn)
print("\n--- Patients per Department ---")
print(department_counts.to_markdown(index=False))

# b. Filtering (e.g., patients in 'Healthcare' department)
query_filter_healthcare = "SELECT * FROM patients WHERE department = 'Healthcare'"
display_data(query_filter_healthcare, "Patients in Healthcare Department")

# c. All patients
query_all = "SELECT * FROM patients"
display_data(query_all, "All Patients Before Deletion")

# 5. Delete a specific patient record (e.g., 'John Doe')
delete_name = 'John Doe'
cursor.execute("DELETE FROM patients WHERE name = ?", (delete_name,))
conn.commit()
print(f"\nDeleted patient: {delete_name}")

# 6. Display remaining patients to show changes
display_data(query_all, "All Patients After Deletion")

# Close the connection
conn.close()
print("\nDatabase connection closed.")

Database and 'patients' table created successfully.

Sample patient records inserted.

--- Database Statistics ---
Total Patients: 11
Average Age: 49.73 years
Average Bill: $2463.85
Bill Range: $800.00 - $5500.00

--- Patients per Department ---
| department   |   patient_count |
|:-------------|----------------:|
| Cardiology   |               3 |
| Healthcare   |               4 |
| Neurology    |               2 |
| Oncology     |               2 |

--- Patients in Healthcare Department ---
|   id | name        |   age | department   |   bill |
|-----:|:------------|------:|:-------------|-------:|
|    3 | Peter Jones |    28 | Healthcare   |    800 |
|    5 | Mike Brown  |    42 | Healthcare   |   1200 |
|    9 | Peter Jones |    28 | Healthcare   |    800 |
|   11 | Mike Brown  |    42 | Healthcare   |   1200 |

--- All Patients Before Deletion ---
|   id | name        |   age | department   |    bill |
|-----:|:------------|------:|:-------------|--------:|
|    2 | Jane Smith  